# Expression prediction across genes and cell types

This notebook loads a trained sequence-to-expression model, prepares genomic sequences around annotated transcription start sites, pairs each sequence with cell-type metadata, predicts expression, and compares the predictions with held-out measurements.

## Imports

Import filesystem utilities, plotting and table libraries, parallel-processing helpers, and the expression-model API. The final three imports provide the core abstractions used below:

- `SequenceModel` loads and runs the trained model.
- `Genome` retrieves annotated DNA sequences from a reference genome.
- `Condition` and `DescriptionLookup` associate each cell type with its metadata description.

In [1]:
import jax
print(jax.__version__)
print(jax.devices())

import os
#os.environ['CUDA_VISIBLE_DEVICES'] = '1'
#import jax
from alphagenome.data import gene_annotation
from alphagenome.data import genome
from alphagenome.data import transcript as transcript_utils
from alphagenome.visualization import plot_components
from alphagenome_research.model import dna_model
from alphagenome.models.variant_scorers import CenterMaskScorer
from alphagenome.models.variant_scorers import AggregationType
from alphagenome.models import variant_scorers
from alphagenome.models import dna_client

0.10.2
[CudaDevice(id=0), CudaDevice(id=1), CudaDevice(id=2), CudaDevice(id=3), CudaDevice(id=4), CudaDevice(id=5), CudaDevice(id=6)]


/home/jovyan/miniconda3/envs/api/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
I0000 00:00:1788428103.450013  676001 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1788428107.443737  676001 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
import os
from pathlib import Path
import polars as pl
from concurrent.futures import ProcessPoolExecutor
from functools import partial
import tqdm
import numpy as np
import pandas as pd

import gena_expression as gx
from gena_expression.conditions import Condition, DescriptionLookup
from gena_expression.dna import Genome
from gena_expression.inference import SequenceModel, AlphaGenomeSequenceModel

from gena_expression.scoring.scorers import TrackWindowScorer

## Configure paths and inference settings

Define the locations of model code, checkpoint, configuration, tokenizers, genome resources, and metadata. These paths are environment-specific and should be updated when running the notebook elsewhere.

The settings also define the GPU device and the number of DNA tokens fetched upstream of each transcription start site.

In [3]:
# Paths and constants used by later cells

from pathlib import Path

GENALM_HOME = Path("/workspace-SR003.nfs2/estsoi/CAGI5_benchmark")

os.environ["GENALM_HOME"] = GENALM_HOME.__str__()

GENA_LM_ROOT = GENALM_HOME / "GENA_LM" # CHANGE THIS LINE

EXPRESSION_TASK_DIR = GENA_LM_ROOT / "downstream_tasks" / "expression_prediction"

MODEL_FILE = EXPRESSION_TASK_DIR / "expression_model_final.py"
MODEL_CLS = f"{MODEL_FILE}::ExpressionCounts"

MODEL_CHECKPOINT = GENALM_HOME / "models" / "expression_model_v1-1" / "pytorch_model.bin" # CHANGE THIS LINE
MODEL_CONFIG = EXPRESSION_TASK_DIR / 'configs' / "final.yaml" # CHANGE THIS LINE
DNA_TOKENIZER = GENA_LM_ROOT / "data" / "tokenizers" / "t2t_1000h_multi_32k"
DESCRIPTION_TOKENIZER = "Qwen/Qwen3-Embedding-0.6B"

DATA_DIR = Path("../data").resolve()
DESCRIPTION_DIR = DATA_DIR / "real_descriptions"

GENOME_FASTA = Path("/home/jovyan/.cache/mpramnist/data/Kircher/hg38.fa") # CHANGE THIS LINE
GENOME_ANNOTATION = Path(
    "/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/"
    "downstream_tasks/expression_prediction/datasets/data/genomes/hg38/"
    "gencode.v29.primary_assembly.annotation_UCSC_names.gtf"
) # CHANGE THIS LINE

DEVICE = "cuda:5" # CHANGE THIS LINE


TOKEN_LEN_FOR_FETCH = 15
NUM_BEFORE = 512
DNA_MAX_SEQ_LEN = 1024
DESC_MAX_SEQ_LEN = 510

## Load the trained sequence model

Load the expression-prediction model together with its checkpoint, model configuration, DNA tokenizer, and description tokenizer.

The sequence-length settings specify how DNA and metadata are represented at inference time. The model is placed on the selected GPU device.

In [4]:
model_ag = dna_model.create_from_huggingface('all_folds', device=jax.devices()[0], organism_settings={dna_model.Organism.HOMO_SAPIENS: dna_model.OrganismSettings(fasta_path='/home/jovyan/.cache/mpramnist/data/Kircher/hg38.fa'), 
                                                                                                   dna_model.Organism.MUS_MUSCULUS: (
            dna_model.OrganismSettings()
        )})

Fetching 12 files: 100%|██████████| 12/12 [00:00<00:00, 13812.20it/s]


In [ ]:
model = SequenceModel.load(
    model_cls=MODEL_CLS,
    checkpoint=MODEL_CHECKPOINT,
    config=MODEL_CONFIG,
    dna_tokenizer=DNA_TOKENIZER,
    description_tokenizer=DESCRIPTION_TOKENIZER,
    dna_max_seq_len=DNA_MAX_SEQ_LEN,
    num_before=NUM_BEFORE,
    desc_max_seq_len=DESC_MAX_SEQ_LEN,
    token_len_for_fetch=TOKEN_LEN_FOR_FETCH,
    device=DEVICE,
)

/home/jovyan/miniconda3/envs/api/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace-SR003.nfs2/estsoi/API/GENA_LM/downstream_tasks/expression_prediction/api/src/gena_expression/inference/model.py:162: UserWarning: 
The version_base parameter is not specified.
Please specify a compatability version level, or None.
Will assume defaults for version 1.1
  with initialize_config_dir(str(config_path.parents[0])):


Using ModernGENA from AIRI-Institute/moderngena-large


Flash Attention 2 only supports torch.float16 and torch.bfloat16 dtypes, but the current dype in ModernBertModel is torch.float32. You should run training or inference using Automatic Mixed-Precision via the `with torch.autocast(device_type='torch_device'):` decorator, or load the model with the `torch_dtype` argument. Example: `model = AutoModel.from_pretrained("openai/whisper-tiny", attn_implementation="flash_attention_2", torch_dtype=torch.float16)`


missing: 0 []
unexpected: 3 ['decoder.bias', 'head.dense.weight', 'head.norm.weight']
mismatched: []
bert dropouts: {'attention_dropout': 0.1, 'embedding_dropout': 0.1, 'mlp_dropout': 0.1}
qwen dropouts: {'attention_dropout': 0.1}
[desc_model] unfrozen transformer blocks: [24, 25, 26, 27] (total blocks=28)
[desc_model] backbone.norm trainable: True (trainable params=1,024)
[desc_model] trainable params: 62,924,800 / 595,776,512
[desc_model] trainable tensors: 45
  - layers.24.self_attn.q_proj.weight
  - layers.24.self_attn.k_proj.weight
  - layers.24.self_attn.v_proj.weight
  - layers.24.self_attn.o_proj.weight
  - layers.24.self_attn.q_norm.weight
  - layers.24.self_attn.k_norm.weight
  - layers.24.mlp.gate_proj.weight
  - layers.24.mlp.up_proj.weight
  - layers.24.mlp.down_proj.weight
  - layers.24.input_layernorm.weight
  - layers.24.post_attention_layernorm.weight
  - layers.25.self_attn.q_proj.weight
  - layers.25.self_attn.k_proj.weight
  - layers.25.self_attn.v_proj.weight
  - l

In [5]:
model = AlphaGenomeSequenceModel(model=model_ag,
                                    dna_tokenizer=DNA_TOKENIZER,
                                        output_type=dna_model.OutputType.DNASE,
                                        dna_max_seq_len=DNA_MAX_SEQ_LEN,
                                        num_before=NUM_BEFORE,
                                        token_len_for_fetch=TOKEN_LEN_FOR_FETCH,)

## Load strand-specific gene intervals

Read the validation gene intervals for forward and reverse orientations. Each table provides genomic coordinates for a gene, including its chromosome, transcription start site (TSS), transcription end site (TES), and strand.

In [6]:
intervals_dir = '/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/intervals'
forward_intervals = pl.read_csv(f'{intervals_dir}/human.valid.forward.csv', separator='\t')
reverse_intervals = pl.read_csv(f'{intervals_dir}/human.valid.reverse.csv', separator='\t')

## Initialize genome access and cell-type metadata

Create a `Genome` object for retrieving DNA and genomic annotations from the reference genome.

Create a `DescriptionLookup` from the selected metadata files. Each identifier represents one cell type or experimental condition whose description will be supplied to the model.

In [7]:
genome = Genome(
    fasta_path=GENOME_FASTA,
    annotation=GENOME_ANNOTATION,
)
lookup = DescriptionLookup(
    json_dir=DESCRIPTION_DIR,
    keys=['ENCFF035CWS',
            'ENCFF083EOC',
            'ENCFF123KIW',
            'ENCFF236XOK',
            'ENCFF242BWW',
            'ENCFF329ENM',
            'ENCFF361XCF',
            'ENCFF494KRC',
            'ENCFF602HCV',
            'ENCFF660EXG',
            'ENCFF664WLU',
            'ENCFF761SPP',
            'ENCFF784MDF',
            'ENCFF857JQM']
)

/workspace-SR003.nfs2/estsoi/API/GENA_LM/downstream_tasks/expression_prediction/api/src/gena_expression/dna/genome.py:437: RuntimeWarning: Plain GTF/GFF annotations are not supported directly for fast interval queries; using existing compressed annotation /workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz. debug={'plain_path': '/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/gencode.v29.primary_assembly.annotation_UCSC_names.gtf', 'compressed_path': '/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/GENA_LM/downstream_tasks/expression_prediction/datasets/data/genomes/hg38/gencode.v29.primary_assembly.annotation_UCSC_names.gtf.gz'}
  return _TabixGtfAnnotation(cls._ensure_tabix_gtf(path))
/workspace-SR003.nfs2/estsoi/API/GENA_LM/downstream_tasks/expression_prediction/api/src/gena_

## Build conditions and coordinate lookup tables

Convert every metadata entry into a `Condition` object containing a human-readable name and its description.

Create dictionaries mapping each gene identifier to its chromosome, TSS, TES, and strand separately for forward and reverse interval sets. These dictionaries are used to generate the corresponding DNA inputs efficiently.

In [8]:
cellType2condition_assey = {
'ENCFF035CWS' : 'EFO:0005721 DNase-seq',
'ENCFF083EOC' : "CL:0002351 DNase-seq",
'ENCFF123KIW' : 'CL:1000507 DNase-seq',
'ENCFF236XOK' : 'UBERON:0002369 DNase-seq',
'ENCFF242BWW' : 'UBERON:0002048 DNase-seq',
'ENCFF329ENM' : 'UBERON:0001157 DNase-seq',
'ENCFF361XCF' : 'UBERON:0000945 DNase-seq',
'ENCFF494KRC' : 'UBERON:0001264 DNase-seq',
'ENCFF602HCV' : 'EFO:0002713 DNase-seq',
'ENCFF660EXG' : 'EFO:0001187 DNase-seq',
'ENCFF664WLU' : 'UBERON:0001264 DNase-seq',
'ENCFF761SPP' : 'EFO:0001187 DNase-seq',
'ENCFF784MDF' : 'UBERON:0001115 DNase-seq',
'ENCFF857JQM' : 'UBERON:0000945 DNase-seq'
}
cellType2condition = {k: v.split(' ')[0] for k,v in cellType2condition_assey.items()}
tss_tes_coords_forward = {row['gene_id'] : (row['chrom'], row['TSS'], row['TES'], row['gene_strand']) for row in forward_intervals.iter_rows(named=True)}
tss_tes_coords_reverse = {row['gene_id'] : (row['chrom'], row['TSS'], row['TES'], row['gene_strand']) for row in reverse_intervals.iter_rows(named=True)}

## Fetch one annotated sequence around a gene

Define a worker function that retrieves the genomic DNA sequence for one gene. The requested interval includes a fixed amount of context upstream of the TSS in the model’s reading direction.

The returned sequence retains genomic features and receives an explicit `tss` feature. This feature is later used to align inference windows at the transcription start site.

In [9]:
def _get_sequence(item, genome, reverse, token_len_for_fetch, num_before):
    gene, (chrom, tss, tes, strand) = item

    start, end = (
        (tes, tss + token_len_for_fetch * num_before)
        if reverse
        else (tss - token_len_for_fetch * num_before, tes)
    )

    return genome.sequence(
        chrom=chrom,
        start=start,
        end=end,
        strand=strand,
        include_features=True,
        name=gene,
    ).add_feature(name='tss',
                    type='tss', 
                    start=token_len_for_fetch*num_before, 
                    end=token_len_for_fetch*num_before)

## Generate sequence–condition pairs in parallel

Define a helper that retrieves all gene sequences with multiple worker processes and then pairs every gene with every requested cell type.

For each cell type, the same ordered gene sequence collection is reused while the associated `Condition` changes. The result is a complete Cartesian product of genes and cell-type descriptions, ready for batched inference.

In [10]:
def make_sequences_and_conditions(
    genome: Genome,
    cell_types:list|tuple,
    cellType2condition:dict,
    tss_tes_coords,
    reverse:bool,
    max_workers:int,
    num_before:int,
    token_len_for_fetch:int
):
    cell_types = list(cell_types)
    coordinate_items = list(tss_tes_coords.items())

    worker = partial(
        _get_sequence,
        genome=genome,
        reverse=reverse,
        num_before=num_before, 
        token_len_for_fetch=token_len_for_fetch
    )

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        gene_sequences = list(
            tqdm.tqdm(
                executor.map(
                    worker,
                    coordinate_items,
                    chunksize=100,
                ),
                total=len(coordinate_items),
                desc="Fetching sequences",
            )
        )

    sequences = gene_sequences * len(cell_types)

    conditions = [
        cellType2condition[cell_type]
        for cell_type in cell_types
        for _ in gene_sequences
    ]

    return sequences, conditions

## Build forward- and reverse-orientation inputs

Generate annotated gene sequences for the forward and reverse interval sets and pair them with all cell-type conditions.

Parallel sequence retrieval substantially reduces preprocessing time for large gene collections.

In [11]:
sequences_forward, conditions_forward = make_sequences_and_conditions(genome=genome,
                                                            cell_types=cellType2condition.keys(),
                                                            cellType2condition=cellType2condition,
                                                            tss_tes_coords=tss_tes_coords_forward,
                                                            reverse=False,
                                                            max_workers=10,
                                                            num_before=512,
                                                            token_len_for_fetch=15
                                                            )

sequences_reverse, conditions_reverse = make_sequences_and_conditions(genome=genome,
                                                            cell_types=cellType2condition.keys(),
                                                            cellType2condition=cellType2condition,
                                                            tss_tes_coords=tss_tes_coords_reverse,
                                                            reverse=True,
                                                            max_workers=10,
                                                            num_before=512,
                                                            token_len_for_fetch=15
                                                            )

/home/jovyan/miniconda3/envs/api/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
Fetching sequences: 100%|██████████| 1548/1548 [00:03<00:00, 505.26it/s]/home/jovyan/miniconda3/envs/api/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()

Fetching sequences: 100%|██████████| 1490/1490 [00:04<00:00, 355.46it/s]


## Reference implementation: serial sequence construction

This cell shows the equivalent nested-loop implementation for constructing forward and reverse inputs without parallel processing.

It is retained as a transparent reference for the input construction logic, but is slower than `make_sequences_and_conditions` for large datasets.

In [ ]:
#SLOW, same as code above, DO NOT RUN!
conditions_forward=[]
sequences_forward = []
for cell_type in lookup.keys():
    for gene, (chrom, tss, tes, strand) in tqdm.tqdm(tss_tes_coords_forward.items()):
        sequence = genome.sequence(
                chrom=chrom,
                start=tss-15*512, #token_len_for_fetch * num_before
                end=tes,
                strand=strand,
                include_features=True,
                name=gene,
            ).add_feature(name='tss',
                    type='tss', 
                    start=15*512, 
                    end=15*512)
        sequences_forward.append(sequence)
        conditions_forward.append(cellType2condition[cell_type])

conditions_reverse=[]
sequences_reverse = []
for cell_type in lookup.keys():
    for gene, (chrom, tss, tes, strand) in tqdm.tqdm(tss_tes_coords_reverse.items()):
        sequence = genome.sequence(
                chrom=chrom,
                start=tes, 
                end=tss+15*512, #token_len_for_fetch * num_before
                strand=strand, # strand == "-" would take reverse complement
                include_features=True,
                name=gene,
            ).add_feature(name='tss',
                    type='tss', 
                    start=15*512, 
                    end=15*512)
        sequences_reverse.append(sequence)
        conditions_reverse.append(cellType2condition[cell_type])

100%|██████████| 1490/1490 [00:03<00:00, 436.12it/s]


## Combine all inference inputs

Concatenate forward and reverse gene sequences, along with their matching conditions, into one collection for model inference.

In [12]:
sequences = sequences_forward + sequences_reverse
conditions = conditions_forward + conditions_reverse

## Predict expression for every gene–condition pair

Run batched expression prediction across all prepared sequences and cell-type conditions.

Predictions are centered on the explicitly annotated `tss` feature. Each prediction is converted into a compact record containing the cell type, gene identifier, and scalar expression value.

In [13]:
results = []
predictions = model.predict_multiple_sequences(
    sequences=sequences,
    conditions=conditions,
    center='tss',
    grouping='no_grouping',
    return_tokens=True,
    preprocessing_workers=10,
    prefetch_batches=2,
    max_records_per_forward=400,
)

Predicting AlphaGenome sequences:   0%|          | 0/42532 [00:00<?, ?item/s]

Predicting AlphaGenome sequences: 100%|██████████| 42532/42532 [1:15:41<00:00,  9.37item/s]  


In [25]:
predictions[0].condition.description

'EFO:0005721'

In [16]:
import tqdm

In [ ]:
#OPTIONAL, DO NOT RUN
results = []
scorer = TrackWindowScorer(center='tss', aggregate='sum', width_bp=501)

for prediction in tqdm.tqdm(predictions):
    results.append(
        (
            prediction.condition.description,
            prediction.sequence_name,
            scorer.score_prediction(prediction).score,
        )
    )

100%|██████████| 42532/42532 [02:05<00:00, 337.77it/s]


In [31]:
results = [res for res in results if res[0] != 'condition']

## Create a gene-by-cell-type prediction matrix

Convert the prediction records into a tabular data frame, then pivot it so that rows represent genes and columns represent cell types.

This matrix format matches the layout used for the ground-truth expression measurements.

In [36]:
#df_predicted = pl.DataFrame(results, schema={'Cell Type':pl.String, 'Gene':pl.String, 'Expression':pl.Float64})
df_predicted = pl.DataFrame(results, schema={'Cell Type':pl.String, 'Gene':pl.String, 'Expression':pl.Float64})
df_predicted.columns = ['Cell Type', 'Gene', 'Expression']
df_predicted = df_predicted.unique(['Cell Type', 'Gene']).pivot(on='Cell Type', index='Gene')

/tmp/ipykernel_676001/2749856391.py:2: DataOrientationWarning: Row orientation inferred during DataFrame construction. Explicitly specify the orientation by passing `orient="row"` to silence this warning.
  df_predicted = pl.DataFrame(results, schema={'Cell Type':pl.String, 'Gene':pl.String, 'Expression':pl.Float64})


In [35]:
df_predicted.unique(['Cell Type', 'Gene'])

Cell Type,Gene,Expression
str,str,f64
"""CL:0002351""","""ENSG00000174885.12""",47.877319
"""UBERON:0001264""","""ENSG00000168395.15""",1687.601647
"""CL:1000507""","""ENSG00000101323.4""",13.970033
"""UBERON:0002369""","""ENSG00000246526.2""",45.354373
"""UBERON:0001264""","""ENSG00000078902.15""",2002.677048
…,…,…
"""EFO:0002713""","""ENSG00000125872.7""",178.53103
"""UBERON:0002369""","""ENSG00000185594.5""",22.918868
"""EFO:0005721""","""ENSG00000204104.11""",469.234077


In [37]:
df_predicted.sort(by='Gene')

Gene,EFO:0001187,EFO:0005721,UBERON:0001157,UBERON:0002369,EFO:0002713,UBERON:0001264,CL:1000507,UBERON:0002048,UBERON:0000945,UBERON:0001115,CL:0002351
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000001617.11""",60.449933,76.185492,518.892968,160.774096,24.539632,593.532568,101.41211,240.530688,451.521595,279.043698,406.494874
"""ENSG00000002016.17""",139.241195,78.410773,63.271944,124.066816,328.115433,35.160471,296.412245,146.208306,37.170902,50.204134,39.065548
"""ENSG00000002549.12""",860.066896,814.443968,3746.696774,2457.928224,932.442384,2911.457037,592.381594,2467.590835,2457.363766,1669.051259,1139.756587
"""ENSG00000002587.9""",10.97625,21.907928,54.720795,27.952998,20.38697,59.44123,37.544495,41.588532,45.912421,24.005449,53.750046
"""ENSG00000003393.14""",836.501564,740.006304,1779.572639,1347.080448,1058.86448,1159.76779,658.83074,1541.17435,1139.616087,758.165806,592.627755
…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000285901.1""",47.559902,271.693741,643.089786,580.015024,211.587944,318.295297,209.175608,1040.227354,321.498736,160.669501,173.305082
"""ENSG00000285971.1""",41.296178,51.935669,51.055096,277.883299,81.367584,66.973694,56.219849,161.308879,58.32977,46.10822,45.11638
"""ENSG00000285972.1""",18.24399,9.210242,97.541123,16.167207,4.141927,35.817557,173.904657,8.401317,39.068421,33.668646,28.860266


In [38]:
ground_true = pl.read_csv('/workspace-SR003.nfs2/estsoi/scRNA_seq_experiments/v1_ground_true.csv')

In [39]:
ground_true.sort('gene_id')

gene_id,ENCFF035CWS,ENCFF083EOC,ENCFF123KIW,ENCFF236XOK,ENCFF242BWW,ENCFF329ENM,ENCFF361XCF,ENCFF494KRC,ENCFF602HCV,ENCFF660EXG,ENCFF664WLU,ENCFF761SPP,ENCFF784MDF,ENCFF857JQM
str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""ENSG00000001617.11""",1.079541,1.109003,2.621794,1.266682,3.045891,0.540609,2.526034,1.884014,3.266901,2.67877,1.053421,2.218972,1.311255,2.637604
"""ENSG00000002016.17""",2.286177,2.088097,1.595354,1.543407,1.178615,2.598605,0.909459,2.293584,1.871749,1.846946,2.037829,2.124429,2.028351,1.698534
"""ENSG00000002549.12""",3.037178,2.973722,3.893732,3.832327,3.886906,3.097516,3.550817,2.588759,2.945033,3.730597,2.922421,3.898263,3.879168,4.093605
"""ENSG00000002587.9""",0.546619,0.256456,2.593309,0.687508,1.325394,0.918205,1.041303,0.168223,0.162103,0.006797,0.333568,0.025348,0.050543,0.348569
"""ENSG00000003393.14""",2.238377,1.759425,2.019705,1.69556,1.892243,1.386311,1.919276,1.646933,2.870818,2.422747,1.912536,2.148924,2.786094,3.025538
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000285901.1""",0.724827,0.195678,0.240694,0.637291,0.688993,0.467591,0.007235,0.002128,0.708695,0.754854,0.35787,0.252319,0.234298,0.022656
"""ENSG00000285971.1""",0.184287,0.015292,0.001311,0.116774,0.001219,0.008034,0.007235,0.002128,0.039518,0.006797,0.004869,0.025348,0.00378,0.022656
"""ENSG00000285972.1""",0.014966,0.015292,0.256666,0.004663,0.012511,0.196246,0.007235,0.002128,0.055581,0.05551,0.004869,0.295898,0.029174,0.022656


In [48]:
seen = set()
cellType2condition = {
    k: v for k, v in cellType2condition.items()
    if v not in seen and not seen.add(v)
}
ground_true.columns = [cellType2condition[col] if col in cellType2condition.keys() else col for col in ground_true.columns]

## Evaluate correlation across genes within each cell type

For every shared cell type, calculate the Pearson correlation between predicted and measured expression across genes.

This metric measures how well the model ranks or separates genes within a particular cellular context.

In [49]:
pred = df_predicted.to_pandas().set_index("Gene")
true = ground_true.to_pandas().set_index("gene_id")

common_genes = true.index.intersection(pred.index)
common_cells = true.columns.intersection(pred.columns)

true = true.loc[common_genes, common_cells]
pred = pred.loc[common_genes, common_cells]

rows = []

for cell in common_cells:
    true_vec = true[cell].astype(float).values
    pred_vec = pred[cell].astype(float).values
    corr = np.corrcoef(true_vec, pred_vec)[0, 1]
    rows.append({
        "cell_type": cell,
        "corr_genes": corr,
    })

result = pd.DataFrame(rows)

In [50]:
result

,cell_type,corr_genes
0,EFO:0005721,0.494798
1,CL:0002351,0.529418
2,CL:1000507,0.494245
3,UBERON:0002369,0.502619
4,UBERON:0002048,0.459471
5,UBERON:0001157,0.503749
6,UBERON:0000945,0.515093
7,UBERON:0001264,0.485549
8,EFO:0002713,0.496597
9,EFO:0001187,0.511584


## Evaluate correlation across cell types within each gene

For every shared gene, calculate the Pearson correlation between predicted and measured expression across cell types.

Genes with too few valid observations or no variation in their measured expression are excluded because correlation is undefined or uninformative in those cases.

In [52]:
gene_corrs = []
skipped_genes = []
for gene in common_genes:
    gene_true = true.loc[gene].astype(float)
    gene_pred = pred.loc[gene].astype(float)

    mask = pd.notna(gene_true) & pd.notna(gene_pred)

    gene_true = gene_true[mask]
    gene_pred = gene_pred[mask]

    if len(gene_true) > 3 and np.std(gene_true) > 0:
        corr = np.corrcoef(gene_true, gene_pred)[0, 1]

        if not np.isnan(corr):
            gene_corrs.append(corr)
        else:
            skipped_genes.append(gene)
    else:
        skipped_genes.append(gene)

mean_corr_cells = np.mean(gene_corrs)
result["corr_cells"] = mean_corr_cells

mean_row = pd.DataFrame([{
    "cell_type": "mean",
    "corr_genes": result["corr_genes"].mean(),
    "corr_cells": mean_corr_cells,
}])

result = pd.concat([result, mean_row], ignore_index=True)

In [54]:
result

,cell_type,corr_genes,corr_cells
0,EFO:0005721,0.494798,0.09062
1,CL:0002351,0.529418,0.09062
2,CL:1000507,0.494245,0.09062
3,UBERON:0002369,0.502619,0.09062
4,UBERON:0002048,0.459471,0.09062
5,UBERON:0001157,0.503749,0.09062
6,UBERON:0000945,0.515093,0.09062
7,UBERON:0001264,0.485549,0.09062
8,EFO:0002713,0.496597,0.09062
9,EFO:0001187,0.511584,0.09062
